In [8]:
import sys
from pathlib import Path

import torch
from torchvision import transforms

this_path = Path(__file__) if '__file__' in globals() else Path("<unknown>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = work_path / Path("../torch-tools")
sys.path.append(str(tools_path))

from datasets import Datasets
from run_manager import RunManager, RunsManager
from trainer import Trainer, MultiTrainer, MergeEnsemble
from modules import CrossEntropyLossT
from hook import HookManager
import utils

from models.resnet_ee import resnet18 as offnet
from models.resnet_git_ee_def import resnet18 as gitnet_def
from models.resnet_git_ee import resnet18 as gitnet
from models.resnet_git_ee_bn import resnet18 as gitnet_bn
from models.resnet_git_ee_test import resnet18 as gitnet_test

fetch_ds = Datasets(root=work_path / "assets/datasets/")

base_train_ds = fetch_ds("cifar10_train")
base_val_ds = fetch_ds("cifar10_val")

train_trans = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ToTensor(), base_train_ds.normalizer()]
val_trans = [transforms.ToTensor(), base_train_ds.normalizer()]

train_ds = base_train_ds.transform(train_trans).in_ndata(128)
val_ds = base_val_ds.transform(val_trans).in_ndata(128)

train_dl = train_ds.loader(batch_size=128)
val_dl = val_ds.loader(batch_size=128)

epochs = 1
max_lr = 0.005
# exp_name = "exp_compare_unify_all"
exp_name = "exp_dbg"


# for net in [gitnet_def]:
for net in [gitnet]:
# for net in [gitnet_def, gitnet, gitnet_bn]:
    network = net(num_classes=train_ds.fetch_classes(), nb_fils=4, ee_groups=64)
    loss_func = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(network.parameters(), lr=max_lr)
    scheduler_t = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0, last_epoch=-1), "epoch")
    device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

    ee_trainer = Trainer(network, loss_func, optimizer, scheduler_t, device)

    run_mgr = RunManager(exc_path=this_path, exp_name=exp_name)
    run_mgr.log_param("model_arc", f"{net.__module__} {net.__name__}")
    run_mgr.log_param("epochs", epochs)
    run_mgr.log_param("ensemble_type", "EE")
    run_mgr.log_param("params", ee_trainer.count_params())
    run_mgr.log_text("model_torchinfo.txt", ee_trainer.torchinfo(dl=train_dl, verbose=1))

    for e in range(epochs):
        with HookManager(ee_trainer.network) as hm:
            hm.register_forward(module=ee_trainer.network.fc[1], name='final_layer_fw', fn=lambda module, input, output: output[0].detach().cpu())
            hm.register_backward(module=ee_trainer.network.fc[1], name='final_layer_bw', fn=lambda module, grad_input, grad_output: grad_output[0].detach().cpu())
            train_loss, train_acc = ee_trainer.train_1epoch(train_dl)
            tsr_ee_fw = hm.get('final_layer_fw')
            tsr_ee_bw = hm.get('final_layer_bw')
        if e == 0:
            run_mgr.log_param("grad", ee_trainer.count_params_with_grad())
        val_loss, val_acc = ee_trainer.val_1epoch(val_dl)
        met_dict = {"epoch": e + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc}
        ee_trainer.printmet(met_dict, e + 1, epochs, itv=epochs/5)
        
        run_mgr.log_metrics(met_dict, step=e + 1)
        run_mgr.log_metrics(ee_trainer.timeinfo(), step=e + 1)
        run_mgr.log_metric("grad_mean", ee_trainer.grad_mean(), step=e + 1)

        run_mgr.ref_stats(step=e + 1, itv=epochs/100, last_step=epochs)
        run_mgr.ref_results(step=e + 1, itv=epochs/100, last_step=epochs)

    trainers = []
    for i in range(64):
        network = net(num_classes=train_ds.fetch_classes(), nb_fils=4)
        # optimizer = torch.optim.Adam(network.parameters(), lr=max_lr)
        # scheduler_t = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0, last_epoch=-1), "epoch")
        device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

        trainer = Trainer(network, loss_func, optimizer, scheduler_t, device)
        trainers.append(trainer)

    me_trainers = MergeEnsemble(trainers, device)
    me_trainers.loss_func = torch.nn.CrossEntropyLoss()
    me_trainers.optimizer = torch.optim.Adam([p for trainer in me_trainers.trainers for p in trainer.network.parameters()], lr=max_lr)
    me_trainers.scheduler_t = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0, last_epoch=-1), "epoch")

    run_mgr = RunManager(exc_path=this_path, exp_name=exp_name)
    run_mgr.log_param("model_arc", f"{net.__module__} {net.__name__}")
    run_mgr.log_param("epochs", epochs)
    run_mgr.log_param("ensemble_type", "ME")
    run_mgr.log_param("params", me_trainers.count_params())
    run_mgr.log_text("model_torchinfo.txt", me_trainers.trainers[0].torchinfo())
    for e in range(epochs):
        with HookManager(me_trainers.trainers[0].network) as hm:
            hm.register_forward(module=me_trainers.trainers[0].network.fc[1], name='final_layer_fw', fn=lambda module, input, output: output[0].detach().cpu())
            hm.register_backward(module=me_trainers.trainers[0].network.fc[1], name='final_layer_bw', fn=lambda module, grad_input, grad_output: grad_output[0].detach().cpu())
            train_loss, train_acc = me_trainers.train_1epoch(train_dl)
            tsr_me_fw = hm.get('final_layer_fw')
            tsr_me_bw = hm.get('final_layer_bw')
        if e == 0:
            run_mgr.log_param("grad", me_trainers.count_params_with_grad())
        val_loss_t, val_acc_t = me_trainers.val_1epoch(val_dl, incl_members=True)
        val_loss = val_loss_t[0]
        val_loss_em = val_loss_t[1]
        val_acc = val_acc_t[0]
        val_acc_em = val_acc_t[1]
        # val_loss, val_acc = me_trainers.val_1epoch(val_dl)
        met_dict = {"epoch": e + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc}
        # met_dict = {"train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc, "val_acc_em[0]": val_acc_em[0], "val_acc_em[1]": val_acc_em[1]}
        me_trainers.printmet(met_dict, e + 1, epochs, itv=epochs/5)
        run_mgr.log_metrics(met_dict, step=e + 1)
        run_mgr.log_metrics(me_trainers.timeinfo(), step=e + 1)
        run_mgr.log_metric("grad_mean", me_trainers.grad_mean(), step=e + 1)

        run_mgr.ref_stats(step=e + 1, itv=epochs/100, last_step=epochs)
        run_mgr.ref_results(step=e + 1, itv=epochs/100, last_step=epochs)



Epoch:   1/   1    train_loss: 2.307225    train_acc: 0.140625    val_loss: 2.430541    val_acc: 0.093750    duration: 306.5ms
Epoch:   1/   1    train_loss: 2.315860    train_acc: 0.078125    val_loss: 2.632134    val_acc: 0.078125    duration: 1.0s


In [9]:
print(len(tsr_ee_fw))
print(tsr_ee_fw[0].shape)
ee_fw = torch.stack(tsr_ee_fw).var().mean().item()
print(f"1 / {1/ee_fw}")

print(len(tsr_ee_bw))
print(tsr_ee_bw[0].shape)
ee_bw = torch.stack(tsr_ee_bw).abs().mean().item()
print(f"1 / {1/ee_bw}")

1
torch.Size([640, 1])
1 / 4.1482488449951
1
torch.Size([128, 640, 1])
1 / 45499.74528589139


In [10]:
print(len(tsr_me_fw))
print(tsr_me_fw[0].shape)
me_fw = torch.stack(tsr_me_fw).abs().mean().item()
print(f"1 / {1/me_fw}")

print(len(tsr_me_bw))
print(tsr_me_bw[0].shape)
me_bw = torch.stack(tsr_me_bw).abs().mean().item()
print(f"1 / {1/me_bw}")



1
torch.Size([10, 1])
1 / 2.66952332906929
1
torch.Size([128, 10, 1])
1 / 45451.09153707559


In [11]:
print(tsr_ee_bw[0][0])
print(tsr_me_bw[0][0])

tensor([[ 1.3175e-05],
        [ 1.1332e-05],
        [ 1.1897e-05],
        [ 1.1869e-05],
        [ 1.2110e-05],
        [ 1.1426e-05],
        [ 1.1540e-05],
        [ 1.2233e-05],
        [ 1.3026e-05],
        [-1.0861e-04],
        [ 1.3175e-05],
        [ 1.1332e-05],
        [ 1.1897e-05],
        [ 1.1869e-05],
        [ 1.2110e-05],
        [ 1.1426e-05],
        [ 1.1540e-05],
        [ 1.2233e-05],
        [ 1.3026e-05],
        [-1.0861e-04],
        [ 1.3175e-05],
        [ 1.1332e-05],
        [ 1.1897e-05],
        [ 1.1869e-05],
        [ 1.2110e-05],
        [ 1.1426e-05],
        [ 1.1540e-05],
        [ 1.2233e-05],
        [ 1.3026e-05],
        [-1.0861e-04],
        [ 1.3175e-05],
        [ 1.1332e-05],
        [ 1.1897e-05],
        [ 1.1869e-05],
        [ 1.2110e-05],
        [ 1.1426e-05],
        [ 1.1540e-05],
        [ 1.2233e-05],
        [ 1.3026e-05],
        [-1.0861e-04],
        [ 1.3175e-05],
        [ 1.1332e-05],
        [ 1.1897e-05],
        [ 1

In [12]:
a = ee_trainer.network
b = me_trainers.trainers[0].network

In [13]:
for k, v in a.named_parameters():
    print(k, v.shape)

conv1.1.weight torch.Size([256, 3, 3, 3])
bn1.weight torch.Size([256])
bn1.bias torch.Size([256])
layer1.0.conv1.weight torch.Size([256, 4, 3, 3])
layer1.0.bn1.weight torch.Size([256])
layer1.0.bn1.bias torch.Size([256])
layer1.0.conv2.weight torch.Size([256, 4, 3, 3])
layer1.0.bn2.weight torch.Size([256])
layer1.0.bn2.bias torch.Size([256])
layer1.1.conv1.weight torch.Size([256, 4, 3, 3])
layer1.1.bn1.weight torch.Size([256])
layer1.1.bn1.bias torch.Size([256])
layer1.1.conv2.weight torch.Size([256, 4, 3, 3])
layer1.1.bn2.weight torch.Size([256])
layer1.1.bn2.bias torch.Size([256])
layer2.0.conv1.weight torch.Size([512, 4, 3, 3])
layer2.0.bn1.weight torch.Size([512])
layer2.0.bn1.bias torch.Size([512])
layer2.0.conv2.weight torch.Size([512, 8, 3, 3])
layer2.0.bn2.weight torch.Size([512])
layer2.0.bn2.bias torch.Size([512])
layer2.0.downsample.0.weight torch.Size([512, 4, 1, 1])
layer2.0.downsample.1.weight torch.Size([512])
layer2.0.downsample.1.bias torch.Size([512])
layer2.1.conv1.w

In [14]:
aa = a.fc[1].weight.grad.abs().mean().item()
bb = b.fc[1].weight.grad.abs().mean().item()
print(bb / aa)

1.126536122062343
